# Multi-Source Weather Comparison for ROS Detection — Arviat, Nunavut

A single weather station gives accurate *point* observations but zero spatial coverage.
A reanalysis grid (ERA5) gives spatial coverage but has known biases in precipitation
amount and phase — especially in Arctic environments.

This notebook combines **three independent sources** to produce a more reliable
Rain-on-Snow (ROS) event record:

| Source | Type | Resolution | Variables |
|--------|------|-----------|-----------|
| **GHCN-Daily** | Station observation | Point | TMAX, TMIN, PRCP, SNOW, SNWD |
| **ERA5-Land** | Reanalysis (ECMWF) | ~9 km | temp, precip, rain, snowfall |
| **ERA5** | Reanalysis (ECMWF) | ~25 km | same (coarser spatial scale) |

### What you will get
- Bias statistics (ERA5 vs station) for temperature and precipitation
- Monthly bias correction applied to ERA5-Land
- ROS events detected by each source independently
- A **consensus event list** ranked by how many sources agree
- All outputs saved to `multisource_output/`

> **Prerequisite:** Run the Arviat ROS demo notebook first (or just run this one — it
> re-downloads its own ERA5 data independently).

---
## 1. Setup

In [ ]:
# Install required packages (safe to re-run)
%pip install -q requests pandas numpy matplotlib scipy

In [ ]:
import time, warnings, re
from io import StringIO
from pathlib import Path
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats

%matplotlib inline
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

OUT = Path("multisource_output")
OUT.mkdir(exist_ok=True)
print("Output folder:", OUT.resolve())

### Configuration

Edit only this cell.

In [ ]:
# ── Edit here ─────────────────────────────────────────────────────────────────
ARVIAT_CONFIG = {
    "lat":        61.108,
    "lon":        -94.058,
    "name":       "Arviat, Nunavut",
    "station_radius_km": 150,     # search radius for GHCN stations
    "date_range": ["2010-01-01", "2024-12-31"],
}
# ─────────────────────────────────────────────────────────────────────────────

WINTER_MONTHS  = [10, 11, 12, 1, 2, 3, 4, 5]   # Oct–May (snow season)
ROS_TMAX_THRESH = 0.0    # °C
ROS_PRCP_THRESH = 0.0    # mm

LAT = ARVIAT_CONFIG["lat"]
LON = ARVIAT_CONFIG["lon"]
START, END = ARVIAT_CONFIG["date_range"]

print(f"Location : {ARVIAT_CONFIG['name']}  ({LAT}N, {abs(LON):.3f}W)")
print(f"Period   : {START} -> {END}")

---
## 2. GHCN Station Data

We first search the NOAA GHCN station inventory to find weather stations
within **150 km** of Arviat, then download their daily records.

The GHCN-Daily dataset provides *actual ground observations* — temperature
measured by a thermometer, precipitation caught in a rain gauge.
This is the closest thing to "ground truth" we have.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    '''Great-circle distance in km between two points.'''
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def find_ghcn_stations(lat, lon, radius_km=150):
    '''
    Download the GHCN station inventory and return all stations within
    radius_km of (lat, lon), sorted by distance.
    '''
    url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
    print("Downloading GHCN station inventory (~1 MB)...")
    r = requests.get(url, timeout=90)
    r.raise_for_status()

    stations = []
    for line in r.text.splitlines():
        try:
            sid  = line[0:11].strip()
            slat = float(line[12:20])
            slon = float(line[21:30])
            selev = float(line[31:37])
            sname = line[41:71].strip()
            d = haversine(lat, lon, slat, slon)
            if d <= radius_km:
                stations.append({"id": sid, "name": sname,
                                 "lat": slat, "lon": slon,
                                 "elev_m": selev, "dist_km": round(d, 1)})
        except Exception:
            pass

    return sorted(stations, key=lambda x: x["dist_km"])

In [ ]:
nearby = find_ghcn_stations(LAT, LON, ARVIAT_CONFIG["station_radius_km"])

if not nearby:
    print(f"No GHCN stations found within {ARVIAT_CONFIG['station_radius_km']} km.")
    print("Try increasing station_radius_km in the config, or skip to Section 3 (ERA5 only).")
else:
    print(f"Found {len(nearby)} station(s):\n")
    df_stations = pd.DataFrame(nearby)
    display(df_stations[["id","name","lat","lon","elev_m","dist_km"]]
            .rename(columns={"id":"Station ID","name":"Name","lat":"Lat","lon":"Lon",
                             "elev_m":"Elev (m)","dist_km":"Dist (km)"}))

In [ ]:
def download_ghcn(station_id, start_year, end_year, cache_path=None):
    '''
    Download and clean GHCN-Daily CSV for one station.
    Returns a DataFrame with columns:
        date, tmax_c, tmin_c, prcp_mm, snow_mm, snwd_mm
    '''
    if cache_path and Path(cache_path).exists():
        print(f"  Using cached: {cache_path}")
        return pd.read_csv(cache_path, parse_dates=["date"])

    url = (f"https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily"
           f"/access/{station_id}.csv")
    print(f"  Downloading {station_id} ...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()

    raw = pd.read_csv(StringIO(r.text), parse_dates=["DATE"], low_memory=False)
    raw = raw.rename(columns={"DATE": "date"})
    raw = raw[(raw["date"].dt.year >= start_year) &
              (raw["date"].dt.year <= end_year)].copy()

    df = pd.DataFrame({"date": raw["date"]})

    def extract(col, scale=1.0):
        if col not in raw.columns:
            return np.nan
        s = pd.to_numeric(raw[col], errors="coerce")
        # GHCN flags: -9999 = missing
        s[s == -9999] = np.nan
        # Quality flag: drop flagged rows
        qflag = col + "_ATTRIBUTES"
        if qflag in raw.columns:
            bad = raw[qflag].astype(str).str[1] != " "
            s[bad] = np.nan
        return s.values * scale

    # TMAX/TMIN in tenths of °C; PRCP in tenths of mm; SNOW/SNWD in mm
    df["tmax_c"]  = extract("TMAX", scale=0.1)
    df["tmin_c"]  = extract("TMIN", scale=0.1)
    df["prcp_mm"] = extract("PRCP", scale=0.1)
    df["snow_mm"] = extract("SNOW", scale=1.0)
    df["snwd_mm"] = extract("SNWD", scale=1.0)

    if cache_path:
        df.to_csv(cache_path, index=False)
        print(f"  Saved: {cache_path}")
    return df

In [ ]:
# Download the closest station (and optionally the second-closest)
sy = int(START[:4])
ey = int(END[:4])

ghcn_frames = {}
for stn in nearby[:2]:   # up to 2 stations
    cache = str(OUT / f"ghcn_{stn['id']}.csv")
    try:
        df = download_ghcn(stn["id"], sy, ey, cache_path=cache)
        pct_tmax = df["tmax_c"].notna().mean() * 100
        pct_prcp = df["prcp_mm"].notna().mean() * 100
        print(f"  {stn['name']} ({stn['dist_km']} km)  "
              f"TMAX coverage={pct_tmax:.0f}%  PRCP coverage={pct_prcp:.0f}%")
        if pct_tmax > 30:   # require at least 30% data
            ghcn_frames[stn["id"]] = {"meta": stn, "data": df}
    except Exception as e:
        print(f"  Could not download {stn['id']}: {e}")

if not ghcn_frames:
    print("\nNo usable station data. Proceeding with ERA5 sources only.")
    PRIMARY_STN = None
else:
    PRIMARY_STN = list(ghcn_frames.keys())[0]
    print(f"\nPrimary station: {ghcn_frames[PRIMARY_STN]['meta']['name']}  ({PRIMARY_STN})")

---
## 3. ERA5 Reanalysis Data

We download two ERA5 products from [Open-Meteo](https://open-meteo.com)
(free, no account):

- **ERA5-Land** (~9 km): higher spatial resolution, better for local terrain
- **ERA5** (~25 km): coarser grid, but spatially consistent over large areas

Comparing both against the station shows whether the finer resolution
actually improves the representation of local conditions.


In [ ]:
def fetch_openmeteo(lat, lon, start, end, model, cache_path=None):
    '''
    Download daily weather from Open-Meteo archive for a given model.
    model: 'era5_land' or 'era5'
    Returns a DataFrame with date, tmax_c, tmin_c, prcp_mm, rain_mm, snow_mm
    '''
    if cache_path and Path(cache_path).exists():
        print(f"  [{model}] Using cached data")
        return pd.read_csv(cache_path, parse_dates=["date"])

    print(f"  [{model}] Downloading {start} -> {end} ...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": ("temperature_2m_max,temperature_2m_min,"
                  "precipitation_sum,rain_sum,snowfall_sum"),
        "timezone": "UTC",
        "models": model,
    }
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=90)
            r.raise_for_status()
            break
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"Open-Meteo download failed: {e}")
            time.sleep(5)

    df = pd.DataFrame(r.json()["daily"])
    df["date"] = pd.to_datetime(df["time"])
    df = df.drop(columns=["time"]).rename(columns={
        "temperature_2m_max": "tmax_c",
        "temperature_2m_min": "tmin_c",
        "precipitation_sum":  "prcp_mm",
        "rain_sum":           "rain_mm",
        "snowfall_sum":       "snow_mm",
    })
    if cache_path:
        df.to_csv(cache_path, index=False)
    return df

In [ ]:
era5land = fetch_openmeteo(LAT, LON, START, END, model="era5_land",
                           cache_path=str(OUT / "era5_land.csv"))
era5     = fetch_openmeteo(LAT, LON, START, END, model="era5",
                           cache_path=str(OUT / "era5.csv"))

print(f"\nERA5-Land : {len(era5land):,} days  "
      f"({era5land.date.min().date()} -> {era5land.date.max().date()})")
print(f"ERA5      : {len(era5):,} days")

---
## 4. Align All Sources

Merge all datasets onto a common daily date index so we can compare them
directly. The GHCN station may have gaps — these appear as `NaN`.


In [ ]:
# Common date range = ERA5-Land dates
date_idx = pd.DataFrame({"date": era5land["date"]})

# ERA5-Land
master = date_idx.merge(
    era5land[["date","tmax_c","tmin_c","prcp_mm","snow_mm"]]
    .rename(columns=lambda c: f"el_{c}" if c != "date" else c),
    on="date", how="left"
)
# ERA5
master = master.merge(
    era5[["date","tmax_c","tmin_c","prcp_mm","snow_mm"]]
    .rename(columns=lambda c: f"e5_{c}" if c != "date" else c),
    on="date", how="left"
)
# GHCN primary station (if available)
if PRIMARY_STN:
    stn_df = ghcn_frames[PRIMARY_STN]["data"]
    master = master.merge(
        stn_df[["date","tmax_c","tmin_c","prcp_mm","snow_mm","snwd_mm"]]
        .rename(columns=lambda c: f"stn_{c}" if c != "date" else c),
        on="date", how="left"
    )

master["month"] = master["date"].dt.month
master["year"]  = master["date"].dt.year
master = master.set_index("date")

print(f"Master table: {len(master):,} days,  {len(master.columns)} columns")
print("\nData availability (% non-null):")
avail = {}
for src, col in [("ERA5-Land","el_tmax_c"), ("ERA5","e5_tmax_c")]:
    avail[src] = f"{master[col].notna().mean()*100:.0f}%"
if PRIMARY_STN:
    avail["GHCN station"] = f"{master['stn_tmax_c'].notna().mean()*100:.0f}%"
for k,v in avail.items():
    print(f"  {k:15s}: {v}")

---
## 5. Temperature Comparison

How well do ERA5-Land and ERA5 track the station thermometer?

- **Time series**: all three sources on the same axes
- **Scatter plot**: station vs each reanalysis product
- **Bias stats**: mean error, RMSE, correlation


In [ ]:
if PRIMARY_STN:
    fig = plt.figure(figsize=(14, 9))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    # ── Top: time series (winter months only for clarity) ────────────────────
    ax0 = fig.add_subplot(gs[0, :])
    win = master[master["month"].isin(WINTER_MONTHS)]
    ax0.plot(win.index, win["stn_tmax_c"],  lw=0.7, color="black",    label="GHCN station", alpha=0.9, zorder=3)
    ax0.plot(win.index, win["el_tmax_c"],   lw=0.7, color="steelblue",label="ERA5-Land", alpha=0.75)
    ax0.plot(win.index, win["e5_tmax_c"],   lw=0.7, color="tomato",   label="ERA5 0.25°",  alpha=0.65)
    ax0.axhline(0, color="k", lw=0.8, ls="--", alpha=0.4)
    ax0.set_title("Daily Max Temperature — Winter months (Oct–May)", fontweight="bold")
    ax0.set_ylabel("Tmax (°C)")
    ax0.legend(fontsize=8, ncol=3)
    ax0.grid(alpha=0.2)

    # ── Bottom-left: scatter ERA5-Land vs station ─────────────────────────────
    ax1 = fig.add_subplot(gs[1, 0])
    ok = master[["stn_tmax_c","el_tmax_c"]].dropna()
    ax1.scatter(ok["stn_tmax_c"], ok["el_tmax_c"], s=2, alpha=0.3, color="steelblue")
    lim = [min(ok.min()), max(ok.max())]
    ax1.plot(lim, lim, "k--", lw=1)
    slope, intercept, r, *_ = stats.linregress(ok["stn_tmax_c"], ok["el_tmax_c"])
    bias  = (ok["el_tmax_c"] - ok["stn_tmax_c"]).mean()
    rmse  = np.sqrt(((ok["el_tmax_c"] - ok["stn_tmax_c"])**2).mean())
    ax1.set_xlabel("Station Tmax (°C)"); ax1.set_ylabel("ERA5-Land Tmax (°C)")
    ax1.set_title(f"ERA5-Land vs Station\nr={r:.3f}  bias={bias:+.2f}°C  RMSE={rmse:.2f}°C", fontsize=9)

    # ── Bottom-right: scatter ERA5 vs station ─────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 1])
    ok2 = master[["stn_tmax_c","e5_tmax_c"]].dropna()
    ax2.scatter(ok2["stn_tmax_c"], ok2["e5_tmax_c"], s=2, alpha=0.3, color="tomato")
    lim2 = [min(ok2.min()), max(ok2.max())]
    ax2.plot(lim2, lim2, "k--", lw=1)
    slope2, intercept2, r2, *_ = stats.linregress(ok2["stn_tmax_c"], ok2["e5_tmax_c"])
    bias2 = (ok2["e5_tmax_c"] - ok2["stn_tmax_c"]).mean()
    rmse2 = np.sqrt(((ok2["e5_tmax_c"] - ok2["stn_tmax_c"])**2).mean())
    ax2.set_xlabel("Station Tmax (°C)"); ax2.set_ylabel("ERA5 0.25° Tmax (°C)")
    ax2.set_title(f"ERA5 0.25° vs Station\nr={r2:.3f}  bias={bias2:+.2f}°C  RMSE={rmse2:.2f}°C", fontsize=9)

    fig.suptitle(f"Temperature Comparison — {ARVIAT_CONFIG['name']}", fontsize=12, fontweight="bold")
    fig.savefig(str(OUT / "temp_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    # No station — compare ERA5-Land vs ERA5
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ok = master[["el_tmax_c","e5_tmax_c"]].dropna()
    axes[0].plot(master.index[:365], master["el_tmax_c"][:365], lw=0.8, color="steelblue", label="ERA5-Land")
    axes[0].plot(master.index[:365], master["e5_tmax_c"][:365], lw=0.8, color="tomato", label="ERA5 0.25°")
    axes[0].set_title("ERA5-Land vs ERA5 — first year"); axes[0].legend(); axes[0].grid(alpha=0.2)
    axes[1].scatter(ok["e5_tmax_c"], ok["el_tmax_c"], s=2, alpha=0.3)
    lim = [ok.min().min(), ok.max().max()]
    axes[1].plot(lim, lim, "k--", lw=1)
    axes[1].set_xlabel("ERA5 Tmax"); axes[1].set_ylabel("ERA5-Land Tmax")
    axes[1].set_title("ERA5-Land vs ERA5 scatter")
    fig.suptitle(f"Temperature — {ARVIAT_CONFIG['name']}\n(No station data found)", fontweight="bold")
    plt.tight_layout(); plt.show()
    print("NOTE: No GHCN station found. Install station data for full comparison.")

In [ ]:
if PRIMARY_STN:
    # Monthly mean temperature bias: ERA5-Land − station
    bias_el = (master["el_tmax_c"] - master["stn_tmax_c"]).groupby(master["month"])
    bias_e5 = (master["e5_tmax_c"] - master["stn_tmax_c"]).groupby(master["month"])

    fig, ax = plt.subplots(figsize=(10, 4))
    months  = list(range(1, 13))
    mnames  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    w = 0.35
    bars1 = ax.bar([m - w/2 for m in months], [bias_el.mean().get(m, np.nan) for m in months],
                   width=w, color="steelblue", alpha=0.8, label="ERA5-Land bias")
    bars2 = ax.bar([m + w/2 for m in months], [bias_e5.mean().get(m, np.nan) for m in months],
                   width=w, color="tomato",    alpha=0.8, label="ERA5 0.25° bias")

    # Add ±1 std error bars
    for i, m in enumerate(months):
        for bars, grp in [(bars1, bias_el), (bars2, bias_e5)]:
            std = grp.std().get(m, np.nan)
            mn  = grp.mean().get(m, np.nan)
            if not np.isnan(std) and not np.isnan(mn):
                ax.errorbar(bars[i].get_x() + bars[i].get_width()/2,
                            mn, yerr=std, color="black", capsize=3, lw=1)

    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(months); ax.set_xticklabels(mnames)
    ax.set_xlabel("Month"); ax.set_ylabel("Bias = Reanalysis - Station (°C)")
    ax.set_title("Monthly Temperature Bias (ERA5 products vs GHCN station)\n"
                 "Positive = reanalysis is warmer than station", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig.savefig(str(OUT / "monthly_temp_bias.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Print table
    bias_table = pd.DataFrame({
        "Month": mnames,
        "ERA5-Land bias (°C)": [round(bias_el.mean().get(m, np.nan), 2) for m in months],
        "ERA5 bias (°C)":      [round(bias_e5.mean().get(m, np.nan), 2) for m in months],
    })
    display(bias_table.set_index("Month").T)

---
## 6. Precipitation Comparison

Precipitation is harder to validate than temperature because:
- Rain gauges under-catch snowfall (wind-induced loss can be 20–50%)
- ERA5 outputs precipitation over a ~9 km grid cell, not a gauge point
- Phase partitioning (rain vs snow) differs between models

Despite these caveats, comparing totals and timing reveals systematic biases.


In [ ]:
if PRIMARY_STN:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"Precipitation Comparison — {ARVIAT_CONFIG['name']}", fontweight="bold", fontsize=12)

    # ── Annual totals ─────────────────────────────────────────────────────────
    ax = axes[0, 0]
    ann_el  = master.groupby("year")["el_prcp_mm"].sum()
    ann_e5  = master.groupby("year")["e5_prcp_mm"].sum()
    years = sorted(set(ann_el.index) & set(ann_e5.index))
    ax.plot(years, [ann_el.get(y, np.nan) for y in years], "o-", color="steelblue", label="ERA5-Land", ms=5)
    ax.plot(years, [ann_e5.get(y, np.nan) for y in years], "s-", color="tomato",    label="ERA5 0.25°", ms=5)
    if PRIMARY_STN:
        ann_stn = master.groupby("year")["stn_prcp_mm"].sum()
        ax.plot(years, [ann_stn.get(y, np.nan) for y in years], "^-", color="black", label="Station", ms=5)
    ax.set_title("Annual total precipitation (mm)"); ax.legend(fontsize=8); ax.grid(alpha=0.2)
    ax.set_xlabel("Year"); ax.set_ylabel("mm/year")

    # ── Monthly mean (climatology) ────────────────────────────────────────────
    ax = axes[0, 1]
    mnames = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    for col, label, color in [("el_prcp_mm","ERA5-Land","steelblue"),
                               ("e5_prcp_mm","ERA5 0.25°","tomato"),
                               ("stn_prcp_mm","Station","black")]:
        if col in master.columns:
            clim = master.groupby("month")[col].mean()
            ax.plot(range(1,13), [clim.get(m,np.nan) for m in range(1,13)],
                    "o-", color=color, label=label, ms=5)
    ax.set_xticks(range(1,13)); ax.set_xticklabels(mnames, fontsize=8)
    ax.set_title("Mean daily precip by month (mm/day)"); ax.legend(fontsize=8); ax.grid(alpha=0.2)
    ax.set_xlabel("Month"); ax.set_ylabel("mm/day")

    # ── Scatter: ERA5-Land daily vs station (wet days only) ──────────────────
    ax = axes[1, 0]
    ok = master[["stn_prcp_mm","el_prcp_mm"]].dropna()
    ok = ok[(ok["stn_prcp_mm"] > 0) | (ok["el_prcp_mm"] > 0)]  # at least one reports precip
    ax.scatter(ok["stn_prcp_mm"], ok["el_prcp_mm"], s=3, alpha=0.25, color="steelblue")
    lim = max(ok.max())
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    corr = ok.corr().iloc[0, 1]
    ratio = ok["el_prcp_mm"].sum() / max(ok["stn_prcp_mm"].sum(), 0.001)
    ax.set_xlabel("Station precip (mm)"); ax.set_ylabel("ERA5-Land precip (mm)")
    ax.set_title(f"ERA5-Land vs Station (wet days)\nr={corr:.2f}  ERA5-Land/Station ratio={ratio:.2f}")
    ax.grid(alpha=0.2)

    # ── Precip phase: % rain vs snow by month ─────────────────────────────────
    ax = axes[1, 1]
    el_rain_frac = (master.groupby("month")["el_rain_mm"].sum() /
                    master.groupby("month")["el_prcp_mm"].sum().clip(lower=0.001))
    stn_rain_frac = None
    if "stn_snow_mm" in master.columns:
        stn_tot = master.groupby("month")["stn_prcp_mm"].sum()
        stn_snow = master.groupby("month")["stn_snow_mm"].sum() * 0.1  # snow in mm water equiv (approx)
        stn_rain_frac = ((stn_tot - stn_snow) / stn_tot.clip(lower=0.001)).clip(0, 1)

    ax.bar(range(1,13), [el_rain_frac.get(m,0) for m in range(1,13)],
           color="steelblue", alpha=0.7, label="ERA5-Land rain fraction")
    if stn_rain_frac is not None:
        ax.plot(range(1,13), [stn_rain_frac.get(m,np.nan) for m in range(1,13)],
                "ko-", ms=5, label="Station rain fraction (approx)")
    ax.set_xticks(range(1,13)); ax.set_xticklabels(mnames, fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel("Fraction of precip as rain")
    ax.set_title("Precipitation phase: rain fraction by month\n(1.0 = all rain,  0.0 = all snow)")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(str(OUT / "precip_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for col, label, color, ax in [("el_prcp_mm","ERA5-Land","steelblue",axes[0]),
                                   ("e5_prcp_mm","ERA5 0.25°","tomato",axes[0])]:
        clim = master.groupby("month")[col].mean()
        ax.plot(range(1,13), [clim.get(m,0) for m in range(1,13)], "o-", color=color, label=label)
    axes[0].legend(); axes[0].set_title("Mean daily precip by month")
    ok = master[["el_prcp_mm","e5_prcp_mm"]].dropna()
    axes[1].scatter(ok["e5_prcp_mm"], ok["el_prcp_mm"], s=3, alpha=0.2)
    axes[1].set_xlabel("ERA5 precip"); axes[1].set_ylabel("ERA5-Land precip")
    axes[1].set_title("ERA5 vs ERA5-Land scatter")
    plt.suptitle("Precipitation (no station)", fontweight="bold"); plt.tight_layout(); plt.show()

---
## 7. Bias Correction

We apply a simple **monthly bias correction** to ERA5-Land using the station
as the reference. This adjusts ERA5-Land's estimates to better match local
observations while retaining its spatial coverage advantage.

**Method:**
- **Temperature**: additive correction per month
  `ERA5-Land_corrected = ERA5-Land + (station_mean - ERA5-Land_mean)` by month
- **Precipitation**: multiplicative ratio per month
  `ERA5-Land_corrected = ERA5-Land × (station_total / ERA5-Land_total)` by month

> **Caveat:** This correction is only valid near the station. Do not apply it
> uncritically across the full 50 km study area — use it as an indication of
> the bias magnitude, not a precise spatial correction.


In [ ]:
if PRIMARY_STN:
    # ── Temperature correction: additive monthly offset ───────────────────────
    tmax_bias_by_month = {}
    for m in range(1, 13):
        sub = master[master["month"] == m][["stn_tmax_c","el_tmax_c"]].dropna()
        if len(sub) > 10:
            tmax_bias_by_month[m] = sub["stn_tmax_c"].mean() - sub["el_tmax_c"].mean()
        else:
            tmax_bias_by_month[m] = 0.0

    master["el_tmax_corr"] = master.apply(
        lambda r: r["el_tmax_c"] + tmax_bias_by_month.get(r["month"], 0.0), axis=1)

    # ── Precipitation correction: multiplicative monthly ratio ────────────────
    prcp_ratio_by_month = {}
    for m in range(1, 13):
        sub = master[master["month"] == m][["stn_prcp_mm","el_prcp_mm"]].dropna()
        el_tot  = sub["el_prcp_mm"].sum()
        stn_tot = sub["stn_prcp_mm"].sum()
        if el_tot > 1 and stn_tot > 0:
            prcp_ratio_by_month[m] = stn_tot / el_tot
        else:
            prcp_ratio_by_month[m] = 1.0

    master["el_prcp_corr"] = master.apply(
        lambda r: r["el_prcp_mm"] * prcp_ratio_by_month.get(r["month"], 1.0), axis=1)

    # ── Summary table ─────────────────────────────────────────────────────────
    mnames = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    corr_table = pd.DataFrame({
        "Month": mnames,
        "Tmax offset (°C)": [round(tmax_bias_by_month.get(m, 0), 2) for m in range(1,13)],
        "Precip ratio":     [round(prcp_ratio_by_month.get(m, 1), 2) for m in range(1,13)],
    }).set_index("Month")
    print("Monthly bias corrections applied to ERA5-Land:")
    display(corr_table.T)

    # ── Verification plot: corrected ERA5-Land vs station ─────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("ERA5-Land After Bias Correction vs Station", fontweight="bold")

    ok_t = master[["stn_tmax_c","el_tmax_c","el_tmax_corr"]].dropna()
    axes[0].scatter(ok_t["stn_tmax_c"], ok_t["el_tmax_c"],    s=2, alpha=0.2, color="tomato",    label="Raw ERA5-Land")
    axes[0].scatter(ok_t["stn_tmax_c"], ok_t["el_tmax_corr"], s=2, alpha=0.2, color="steelblue", label="Corrected ERA5-Land")
    lim = [ok_t.min().min(), ok_t.max().max()]
    axes[0].plot(lim, lim, "k--", lw=1)
    r_raw  = ok_t[["stn_tmax_c","el_tmax_c"]].corr().iloc[0,1]
    r_corr = ok_t[["stn_tmax_c","el_tmax_corr"]].corr().iloc[0,1]
    bias_raw  = (ok_t["el_tmax_c"]    - ok_t["stn_tmax_c"]).mean()
    bias_corr = (ok_t["el_tmax_corr"] - ok_t["stn_tmax_c"]).mean()
    axes[0].set_title(f"Temperature\nRaw: r={r_raw:.3f} bias={bias_raw:+.2f}°C  |  "
                      f"Corrected: r={r_corr:.3f} bias={bias_corr:+.2f}°C", fontsize=9)
    axes[0].set_xlabel("Station Tmax (°C)"); axes[0].set_ylabel("ERA5-Land Tmax (°C)")
    axes[0].legend(fontsize=8)

    ok_p = master[(master["stn_prcp_mm"] > 0) | (master["el_prcp_mm"] > 0)][
        ["stn_prcp_mm","el_prcp_mm","el_prcp_corr"]].dropna()
    axes[1].scatter(ok_p["stn_prcp_mm"], ok_p["el_prcp_mm"],    s=3, alpha=0.2, color="tomato",    label="Raw ERA5-Land")
    axes[1].scatter(ok_p["stn_prcp_mm"], ok_p["el_prcp_corr"],  s=3, alpha=0.2, color="steelblue", label="Corrected ERA5-Land")
    lim2 = max(ok_p.max())
    axes[1].plot([0,lim2],[0,lim2],"k--",lw=1)
    r_p_raw  = ok_p[["stn_prcp_mm","el_prcp_mm"]].corr().iloc[0,1]
    r_p_corr = ok_p[["stn_prcp_mm","el_prcp_corr"]].corr().iloc[0,1]
    axes[1].set_title(f"Precipitation (wet days)\nRaw r={r_p_raw:.2f}  |  Corrected r={r_p_corr:.2f}", fontsize=9)
    axes[1].set_xlabel("Station precip (mm)"); axes[1].set_ylabel("ERA5-Land precip (mm)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    fig.savefig(str(OUT / "bias_correction.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No station data available — skipping bias correction.")
    master["el_tmax_corr"]  = master["el_tmax_c"]
    master["el_prcp_corr"]  = master["el_prcp_mm"]
    print("(el_tmax_corr and el_prcp_corr set equal to uncorrected ERA5-Land)")

---
## 8. ROS Event Detection — All Sources

We apply the same 4-rule ROS filter to each source independently:

| Rule | Condition |
|------|-----------|
| 1 | Month ∈ Oct–May |
| 2 | Precipitation > 0 mm |
| 3 | Tmax > 0 °C |
| 4 | Snow on ground (SNWD > 0 mm for station; 14-day rolling snowfall > 5 mm for ERA5) |

**Sources tested:**
- GHCN station (with SNWD snow-depth record)
- ERA5-Land (raw)
- ERA5-Land (bias-corrected)
- ERA5 0.25° (raw)


In [ ]:
def flag_ros_era5(df_col_prcp, df_col_tmax, df_col_snow, month_col):
    '''Flag ROS days from reanalysis: rolling 14-day snow proxy.'''
    roll14 = df_col_snow.rolling(14, min_periods=1).sum()
    return (
        month_col.isin(WINTER_MONTHS) &
        (df_col_prcp > ROS_PRCP_THRESH) &
        (df_col_tmax > ROS_TMAX_THRESH) &
        (roll14 > 5.0)
    )

def flag_ros_station(df_prcp, df_tmax, df_snwd, month_col):
    '''Flag ROS days from station: requires positive snow depth.'''
    return (
        month_col.isin(WINTER_MONTHS) &
        (df_prcp > ROS_PRCP_THRESH) &
        (df_tmax > ROS_TMAX_THRESH) &
        (df_snwd > 0)
    )

# ERA5-Land (raw)
master["ros_el"]     = flag_ros_era5(master["el_prcp_mm"],   master["el_tmax_c"],
                                     master["el_snow_mm"],   master["month"])
# ERA5-Land (corrected)
master["ros_el_corr"] = flag_ros_era5(master["el_prcp_corr"], master["el_tmax_corr"],
                                      master["el_snow_mm"],   master["month"])
# ERA5 0.25°
master["ros_e5"]     = flag_ros_era5(master["e5_prcp_mm"],   master["e5_tmax_c"],
                                     master["e5_snow_mm"],   master["month"])
# GHCN station
if PRIMARY_STN and "stn_snwd_mm" in master.columns:
    master["ros_stn"] = flag_ros_station(master["stn_prcp_mm"], master["stn_tmax_c"],
                                         master["stn_snwd_mm"],  master["month"])
else:
    master["ros_stn"] = False

# Count events per source (group consecutive flagged days into single events)
def count_events(flag_series):
    groups = (flag_series != flag_series.shift()).cumsum()
    return int(flag_series.groupby(groups).first().sum())

print("ROS event counts (unique events, not days):")
for label, col in [("ERA5-Land (raw)",       "ros_el"),
                   ("ERA5-Land (corrected)",  "ros_el_corr"),
                   ("ERA5 0.25°",            "ros_e5"),
                   ("GHCN station",           "ros_stn")]:
    n = count_events(master[col])
    print(f"  {label:25s}: {n} events")

In [ ]:
# ── Agreement calendar ────────────────────────────────────────────────────────
# Count how many sources flag each day as ROS
sources = ["ros_el", "ros_el_corr", "ros_e5", "ros_stn"]
master["ros_count"] = master[sources].sum(axis=1)

# Annual ROS day counts by source
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(f"ROS Event Agreement — {ARVIAT_CONFIG['name']}", fontweight="bold", fontsize=12)

# ── Panel 1: ROS days per year by source ─────────────────────────────────────
ax = axes[0]
ann = master.groupby("year")[sources + ["ros_count"]].sum()
colors_map = {"ros_el":"steelblue","ros_el_corr":"royalblue",
              "ros_e5":"tomato","ros_stn":"black"}
labels_map = {"ros_el":"ERA5-Land (raw)","ros_el_corr":"ERA5-Land (corrected)",
              "ros_e5":"ERA5 0.25°","ros_stn":"Station"}
for col in sources:
    ax.plot(ann.index, ann[col], "o-", ms=4,
            color=colors_map[col], label=labels_map[col])
ax.set_ylabel("ROS days / year")
ax.set_title("Annual ROS day count by source")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)

# ── Panel 2: Agreement heatmap (winter months, last 5 years) ─────────────────
ax = axes[1]
recent = master[master["year"] >= master["year"].max() - 4]
winter_recent = recent[recent["month"].isin(WINTER_MONTHS)].copy()

# Build a 2D array: rows = sources, columns = day index
src_arrays = np.array([winter_recent[c].astype(int).values for c in sources])
im = ax.imshow(src_arrays, aspect="auto", interpolation="none",
               cmap="RdBu_r", vmin=0, vmax=1)
ax.set_yticks(range(4)); ax.set_yticklabels([labels_map[c] for c in sources], fontsize=8)
ax.set_xlabel("Day index (winter months, last 5 years)")
ax.set_title("ROS flags per source — each column is one winter day  (red=flagged, white=not flagged)")

# ── Panel 3: Sources-in-agreement count per event ────────────────────────────
ax = axes[2]
ax.fill_between(winter_recent.index, winter_recent["ros_count"],
                step="mid", alpha=0.6, color="darkorange")
ax.axhline(2, color="gray",  ls="--", lw=1, label="2 sources agree")
ax.axhline(3, color="tomato",ls="--", lw=1, label="3 sources agree")
ax.axhline(4, color="red",   ls="--", lw=1, label="All 4 sources agree")
ax.set_ylim(0, 4.5); ax.set_ylabel("# sources flagging ROS")
ax.set_title("Source agreement — higher = more confident ROS detection")
ax.legend(fontsize=8, ncol=3); ax.grid(alpha=0.2)

plt.tight_layout()
fig.savefig(str(OUT / "ros_agreement.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Consensus Event List

Group consecutive ROS-flagged days into discrete events and score each
by how many sources detected it. Higher confidence = more sources agree.

| Confidence | Meaning |
|-----------|---------|
| 4 / 4 | All sources agree — high confidence |
| 3 / 4 | Three sources agree — moderate-high |
| 2 / 4 | Two sources agree — moderate |
| 1 / 4 | Only one source — treat with caution |


In [ ]:
def extract_events(flag_series, prcp_col, tmax_col, min_gap_days=3):
    '''
    Group consecutive True runs into events, merge runs separated by < min_gap_days.
    Returns DataFrame with start_date, end_date, duration_days, peak_prcp, peak_tmax.
    '''
    events = []
    in_event = False
    start = None
    gap = 0
    for date, row in master[[flag_series, prcp_col, tmax_col]].iterrows():
        flagged = row[flag_series]
        if flagged:
            if not in_event:
                start = date
                in_event = True
            gap = 0
        else:
            if in_event:
                gap += 1
                if gap >= min_gap_days:
                    end = date - pd.Timedelta(days=gap-1)
                    sub = master.loc[start:end]
                    events.append({
                        "start": start.date(),
                        "end": end.date(),
                        "duration_days": (end - start).days + 1,
                        "peak_prcp_mm": round(sub[prcp_col].max(), 1),
                        "peak_tmax_c":  round(sub[tmax_col].max(), 1),
                    })
                    in_event = False; gap = 0
    return pd.DataFrame(events) if events else pd.DataFrame()


# Build event lists per source
ev = {}
src_pairs = [
    ("ros_el",      "el_prcp_mm",   "el_tmax_c",    "ERA5-Land raw"),
    ("ros_el_corr", "el_prcp_corr", "el_tmax_corr", "ERA5-Land corrected"),
    ("ros_e5",      "e5_prcp_mm",   "e5_tmax_c",    "ERA5 0.25deg"),
    ("ros_stn",     "stn_prcp_mm",  "stn_tmax_c",   "Station"),
]
for flag_col, prcp_col, tmax_col, label in src_pairs:
    if prcp_col in master.columns and tmax_col in master.columns:
        df_e = extract_events(flag_col, prcp_col, tmax_col)
        if not df_e.empty:
            ev[label] = df_e

# Consensus: for each ERA5-Land (corrected) event, count how many other sources
# have an event overlapping within 3 days
if "ERA5-Land corrected" in ev:
    ref = ev["ERA5-Land corrected"].copy()
    ref["confidence"] = 0

    for i, row_r in ref.iterrows():
        count = 1   # ERA5-Land corrected itself
        for label, df_e in ev.items():
            if label == "ERA5-Land corrected": continue
            if df_e.empty: continue
            # Check overlap or within 3-day window
            for _, row_e in df_e.iterrows():
                delta_start = abs((pd.Timestamp(row_e["start"]) - pd.Timestamp(row_r["start"])).days)
                if delta_start <= 5:
                    count += 1
                    break
        ref.at[i, "confidence"] = count

    ref = ref.sort_values(["confidence","peak_prcp_mm"], ascending=[False, False])
    ref["confidence_label"] = ref["confidence"].map(
        {4: "High (4/4)", 3: "Moderate-high (3/4)",
         2: "Moderate (2/4)", 1: "Low (1/4)"})

    print(f"Consensus events found: {len(ref)}")
    print(f"  High confidence (3-4 sources): {(ref['confidence'] >= 3).sum()}")
    print(f"  Moderate (2 sources):           {(ref['confidence'] == 2).sum()}")
    print(f"  Low (1 source only):            {(ref['confidence'] == 1).sum()}\n")

    # Display
    display(ref[["start","end","duration_days","peak_prcp_mm","peak_tmax_c",
                 "confidence","confidence_label"]].rename(columns={
        "start":"Start","end":"End","duration_days":"Days",
        "peak_prcp_mm":"Peak precip (mm)","peak_tmax_c":"Peak Tmax (°C)",
        "confidence":"# sources","confidence_label":"Confidence"
    }).style.bar(subset=["# sources"], vmin=0, vmax=4, color="steelblue"))

    # Save
    ref.to_csv(str(OUT / "consensus_ros_events.csv"), index=False)
    print(f"\nSaved: {OUT}/consensus_ros_events.csv")
else:
    print("ERA5-Land corrected events not available for consensus building.")

In [ ]:
# ── Final summary figure: confidence-coloured event timeline ─────────────────
if "ERA5-Land corrected" in ev and "confidence" in ref.columns:
    fig, ax = plt.subplots(figsize=(14, 4))
    conf_colors = {4: "#c0392b", 3: "#e67e22", 2: "#f1c40f", 1: "#bdc3c7"}
    conf_labels_done = set()

    for _, row in ref.iterrows():
        c = int(row["confidence"])
        col = conf_colors[c]
        lbl = row["confidence_label"] if c not in conf_labels_done else "_nolegend_"
        ax.barh(0, width=(pd.Timestamp(row["end"]) - pd.Timestamp(row["start"])).days + 1,
                left=pd.Timestamp(row["start"]), height=0.6,
                color=col, alpha=0.85, label=lbl)
        conf_labels_done.add(c)

    ax.set_yticks([]); ax.set_xlabel("Date")
    ax.set_title(f"ROS Consensus Event Timeline — {ARVIAT_CONFIG['name']}\n"
                 f"({START[:4]}–{END[:4]})  Colour = source agreement",
                 fontweight="bold")
    ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig(str(OUT / "consensus_timeline.png"), dpi=150, bbox_inches="tight")
    plt.show()

---
## 10. Outputs & Next Steps

All outputs are in `multisource_output/`:

```
multisource_output/
├── ghcn_CA....csv                  GHCN station daily data
├── era5_land.csv                   ERA5-Land daily data
├── era5.csv                        ERA5 0.25° daily data
├── consensus_ros_events.csv        Final ranked event list
├── temp_comparison.png             Temperature bias scatter + time series
├── monthly_temp_bias.png           Bias by month (seasonal pattern)
├── precip_comparison.png           Annual totals, phase fractions
├── bias_correction.png             Before/after correction scatter
├── ros_agreement.png               Per-source flags + agreement heatmap
└── consensus_timeline.png          Event timeline coloured by confidence
```

### What to do with the consensus event list

1. **Use high-confidence events** (3–4 sources) for analysis where data quality
   matters most — SAR change detection, community impact assessments.
2. **Investigate moderate-confidence events** (2 sources) — check station logs
   for data gaps; the discrepancy may be a station issue not a real absence.
3. **Treat low-confidence events** (1 source) as hypotheses only — they may be
   real but undetected by coarser data, or they may be noise.

### Recommended next steps

- Feed the consensus event list into the SAR demo notebook
  (`arviat_ros_demo.ipynb`) to process only high-confidence events
- Compare with community knowledge / land-use records to validate
- Extend the station search radius to include Rankin Inlet, Baker Lake
  for cross-station validation of the regional precip signal
